In [ ]:
import os
import glob
import re
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")

MODEL = OLLAMA_MODEL
openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)

In [ ]:
knowledge = {}

# Employees
filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
  name = Path(filename).stem.split(' ')[-1]
  with open(filename, "r", encoding="utf-8") as f:
    knowledge[name.lower()] = f.read()

# Products
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
  name = Path(filename).stem
  with open(filename, "r", encoding="utf-8") as f:
    knowledge[name.lower()] = f.read()

# print(knowledge)
# print(knowledge.keys())
# print(knowledge['rachel_martinez'])

In [ ]:
def get_relevant_context(message):
  words = re.sub(r'[^\w\s]', '', message).lower().split()
  return [knowledge[word] for word in words if word in knowledge]

# get_relevant_context("Who is Lancaster and what is carllm?")

def additional_context(message):
  relevant_context = get_relevant_context(message)
  if not relevant_context:
    result = "There is no additional context relevant to the user's question."
  else:
    result = f"The following additional context might be relevant in answering the user's question:\n\n \
      {relevant_context}"
  return result

# additional_context("Who is Alex Lancaster?")

In [ ]:
SYSTEM_PREFIX = """
  You represent Insurellm, the Insurance Tech company.
  You are an expert in answering questions about Insurellm; its employees and its products.
  You are provided with additional context that might be relevant to the user's question.
  Give brief, accurate answers. If you don't know the answer, say so.

  Relevant context:
"""

def chat(message, history):
  system_message = SYSTEM_PREFIX + additional_context(message)
  messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
  response = openai.chat.completions.create(model=MODEL, messages=messages)
  return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)